# 📄 AI Job Recommender — Full Pipeline
**Colab Notebook**  
Upload your resume (PDF) → AI Summary → Skill Gaps → Roadmap → Live Jobs from LinkedIn & Naukri  
All outputs saved and zipped at the end.

## 🔑 Stage 0 — Set API Keys

In [ ]:
# Stage 0: Set your API keys (run this first)
import os

os.environ["OPENAI_API_KEY"]   = "your-openai-api-key-here"      # 🔑 Replace
os.environ["APIFY_API_TOKEN"]  = "your-apify-api-token-here"     # 🔑 Replace

print("✅ API keys set")

## ⚙️ Stage 1 — Install Dependencies

In [ ]:
# Stage 1: Install all required packages
!pip install -q openai pymupdf python-dotenv apify-client
print("✅ All packages installed")

## 🗂️ Stage 2 — Create Project Structure

In [ ]:
# Stage 2: Create folder structure
import os
os.makedirs("src", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
with open("src/__init__.py", "w") as f:
    f.write("")
print("✅ Folders created: src/  outputs/")

## 📝 Stage 3 — Write `src/helper.py`

In [ ]:
# Stage 3: Write src/helper.py
helper_code = "import fitz  # PyMuPDF\nimport os\nfrom openai import OpenAI\n\nclient = OpenAI(api_key=os.environ[\"OPENAI_API_KEY\"])\n\ndef extract_text_from_pdf(uploaded_file):\n    \\\"\\\"\\\"Extract text from a PDF file path (str) or file-like object.\\\"\\\"\\\"\n    if isinstance(uploaded_file, str):\n        doc = fitz.open(uploaded_file)\n    else:\n        doc = fitz.open(stream=uploaded_file.read(), filetype=\"pdf\")\n    text = \"\"\n    for page in doc:\n        text += page.get_text()\n    return text.strip()\n\ndef ask_openai(prompt, max_tokens=500):\n    \\\"\\\"\\\"Send a prompt to OpenAI gpt-4o and return the response text.\\\"\\\"\\\"\n    response = client.chat.completions.create(\n        model=\"gpt-4o\",\n        messages=[{\"role\": \"user\", \"content\": prompt}],\n        temperature=0.5,\n        max_tokens=max_tokens,\n    )\n    return response.choices[0].message.content\n"

with open("src/helper.py", "w") as f:
    f.write(helper_code)
print("✅ src/helper.py written")

## 🌐 Stage 4 — Write `src/job_api.py`

In [ ]:
# Stage 4: Write src/job_api.py
job_api_code = "from apify_client import ApifyClient\nimport os\n\napify_client = ApifyClient(os.environ[\"APIFY_API_TOKEN\"])\n\ndef fetch_linkedin_jobs(search_query, location=\"india\", rows=60):\n    \\\"\\\"\\\"Fetch LinkedIn job listings via Apify actor.\\\"\\\"\\\"\n    run_input = {\n        \"title\": search_query,\n        \"location\": location,\n        \"rows\": rows,\n        \"proxy\": {\n            \"useApifyProxy\": True,\n            \"apifyProxyGroups\": [\"RESIDENTIAL\"],\n        }\n    }\n    try:\n        run = apify_client.actor(\"BHzefUZlZRKWxkTck\").call(run_input=run_input)\n        return list(apify_client.dataset(run[\"defaultDatasetId\"]).iterate_items())\n    except Exception as e:\n        print(f\"LinkedIn fetch error: {e}\")\n        return []\n\ndef fetch_naukri_jobs(search_query, location=\"india\", rows=60):\n    \\\"\\\"\\\"Fetch Naukri job listings via Apify actor.\\\"\\\"\\\"\n    run_input = {\n        \"keyword\": search_query,\n        \"maxJobs\": rows,\n        \"freshness\": \"all\",\n        \"sortBy\": \"relevance\",\n        \"experience\": \"all\",\n    }\n    try:\n        run = apify_client.actor(\"alpcnRV9YI9lYVPWk\").call(run_input=run_input)\n        return list(apify_client.dataset(run[\"defaultDatasetId\"]).iterate_items())\n    except Exception as e:\n        print(f\"Naukri fetch error: {e}\")\n        return []\n"

with open("src/job_api.py", "w") as f:
    f.write(job_api_code)
print("✅ src/job_api.py written")

## 🔌 Stage 5 — Write `mcp_server.py`

In [ ]:
# Stage 5: Write mcp_server.py
mcp_code = "from mcp.server.fastmcp import FastMCP\nfrom src.job_api import fetch_linkedin_jobs, fetch_naukri_jobs\n\nmcp = FastMCP(\"Job Recommender\")\n\n@mcp.tool()\nasync def fetchlinkedin(listofkey: str):\n    \\\"\\\"\\\"Fetch LinkedIn jobs for given keywords.\\\"\\\"\\\"\n    return fetch_linkedin_jobs(listofkey)\n\n@mcp.tool()\nasync def fetchnaukri(listofkey: str):\n    \\\"\\\"\\\"Fetch Naukri jobs for given keywords.\\\"\\\"\\\"\n    return fetch_naukri_jobs(listofkey)\n\nif __name__ == \"__main__\":\n    mcp.run(transport='stdio')\n"

with open("mcp_server.py", "w") as f:
    f.write(mcp_code)
print("✅ mcp_server.py written")

## 🖥️ Stage 6 — Write `app.py` (Streamlit App)

In [ ]:
# Stage 6: Write app.py
app_code = "import streamlit as st\nfrom src.helper import extract_text_from_pdf, ask_openai\nfrom src.job_api import fetch_linkedin_jobs, fetch_naukri_jobs\n\nst.set_page_config(page_title=\"Job Recommender\", layout=\"wide\")\nst.title(\"\ud83d\udcc4 AI Job Recommender\")\nst.markdown(\"Upload your resume and get job recommendations based on your skills and experience from LinkedIn and Naukri.\")\n\nuploaded_file = st.file_uploader(\"Upload your resume (PDF)\", type=[\"pdf\"])\n\nif uploaded_file:\n    with st.spinner(\"Extracting text from your resume...\"):\n        resume_text = extract_text_from_pdf(uploaded_file)\n\n    with st.spinner(\"Summarizing your resume...\"):\n        summary = ask_openai(f\"Summarize this resume highlighting the skills, education, and experience:\\n\\n{resume_text}\", max_tokens=500)\n\n    with st.spinner(\"Finding Skill Gaps...\"):\n        gaps = ask_openai(f\"Analyze this resume and highlight missing skills, certifications, and experiences needed for better job opportunities:\\n\\n{resume_text}\", max_tokens=400)\n\n    with st.spinner(\"Creating Future Roadmap...\"):\n        roadmap = ask_openai(f\"Based on this resume, suggest a future roadmap to improve this person's career prospects (Skills to learn, certifications needed, industry exposure):\\n\\n{resume_text}\", max_tokens=400)\n\n    st.markdown(\"---\")\n    st.header(\"\ud83d\udcd1 Resume Summary\")\n    st.markdown(f\"<div style='background-color:#000000;padding:15px;border-radius:10px;font-size:16px;color:white;'>{summary}</div>\", unsafe_allow_html=True)\n\n    st.markdown(\"---\")\n    st.header(\"\ud83d\udee0\ufe0f Skill Gaps & Missing Areas\")\n    st.markdown(f\"<div style='background-color:#000000;padding:15px;border-radius:10px;font-size:16px;color:white;'>{gaps}</div>\", unsafe_allow_html=True)\n\n    st.markdown(\"---\")\n    st.header(\"\ud83d\ude80 Future Roadmap & Preparation Strategy\")\n    st.markdown(f\"<div style='background-color:#000000;padding:15px;border-radius:10px;font-size:16px;color:white;'>{roadmap}</div>\", unsafe_allow_html=True)\n\n    st.success(\"\u2705 Analysis Completed Successfully!\")\n\n    if st.button(\"\ud83d\udd0e Get Job Recommendations\"):\n        with st.spinner(\"Extracting keywords...\"):\n            keywords = ask_openai(\n                f\"Based on this resume summary, suggest the best job titles and keywords for searching jobs. Give a comma-separated list only, no explanation.\\n\\nSummary: {summary}\",\n                max_tokens=100\n            )\n            search_keywords_clean = keywords.replace(\"\\n\", \"\").strip()\n\n        st.success(f\"Extracted Job Keywords: {search_keywords_clean}\")\n\n        with st.spinner(\"Fetching jobs from LinkedIn and Naukri...\"):\n            linkedin_jobs = fetch_linkedin_jobs(search_keywords_clean, rows=60)\n            naukri_jobs   = fetch_naukri_jobs(search_keywords_clean, rows=60)\n\n        st.markdown(\"---\")\n        st.header(\"\ud83d\udcbc Top LinkedIn Jobs\")\n        if linkedin_jobs:\n            for job in linkedin_jobs:\n                st.markdown(f\"**{job.get('title')}** at *{job.get('companyName')}*\")\n                st.markdown(f\"- \ud83d\udccd {job.get('location')}\")\n                st.markdown(f\"- \ud83d\udd17 [View Job]({job.get('link')})\")\n                st.markdown(\"---\")\n        else:\n            st.warning(\"No LinkedIn jobs found.\")\n\n        st.markdown(\"---\")\n        st.header(\"\ud83d\udcbc Top Naukri Jobs (India)\")\n        if naukri_jobs:\n            for job in naukri_jobs:\n                st.markdown(f\"**{job.get('title')}** at *{job.get('companyName')}*\")\n                st.markdown(f\"- \ud83d\udccd {job.get('location')}\")\n                st.markdown(f\"- \ud83d\udd17 [View Job]({job.get('url')})\")\n                st.markdown(\"---\")\n        else:\n            st.warning(\"No Naukri jobs found.\")\n"

with open("app.py", "w") as f:
    f.write(app_code)
print("✅ app.py written")

## 📤 Stage 7 — Upload Resume & Run Full Pipeline

In [ ]:
# Stage 7: Upload your PDF resume and run the full pipeline
from google.colab import files
import os, json
from src.helper import extract_text_from_pdf, ask_openai
from src.job_api import fetch_linkedin_jobs, fetch_naukri_jobs

print("📂 Please upload your resume PDF below:")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"✅ Uploaded: {pdf_path}")

# ── Extract text ──
print("\n🔍 Extracting text from resume...")
with open(pdf_path, "rb") as f:
    resume_text = extract_text_from_pdf(f)
print(f"✅ Extracted {len(resume_text)} characters")

with open("outputs/stage1_resume_text.txt", "w") as f:
    f.write(resume_text)
print("💾 Saved → outputs/stage1_resume_text.txt")

In [ ]:
# Stage 7b: Summarize resume
print("🤖 Summarizing resume with GPT-4o...")
summary = ask_openai(
    f"Summarize this resume highlighting the skills, education, and experience:\n\n{resume_text}",
    max_tokens=500
)
print("\n📑 RESUME SUMMARY:\n", summary)

with open("outputs/stage2_summary.txt", "w") as f:
    f.write(summary)
print("\n💾 Saved → outputs/stage2_summary.txt")

In [ ]:
# Stage 7c: Skill gap analysis
print("🔎 Analyzing skill gaps...")
gaps = ask_openai(
    f"Analyze this resume and highlight missing skills, certifications, and experiences needed for better job opportunities:\n\n{resume_text}",
    max_tokens=400
)
print("\n🛠️ SKILL GAPS:\n", gaps)

with open("outputs/stage3_skill_gaps.txt", "w") as f:
    f.write(gaps)
print("\n💾 Saved → outputs/stage3_skill_gaps.txt")

In [ ]:
# Stage 7d: Future roadmap
print("🚀 Creating future roadmap...")
roadmap = ask_openai(
    f"Based on this resume, suggest a future roadmap to improve this person's career prospects (Skills to learn, certifications needed, industry exposure):\n\n{resume_text}",
    max_tokens=400
)
print("\n🗺️ FUTURE ROADMAP:\n", roadmap)

with open("outputs/stage4_roadmap.txt", "w") as f:
    f.write(roadmap)
print("\n💾 Saved → outputs/stage4_roadmap.txt")

In [ ]:
# Stage 7e: Extract job keywords
print("🏷️ Extracting job search keywords...")
keywords = ask_openai(
    f"Based on this resume summary, suggest the best job titles and keywords for searching jobs. Give a comma-separated list only, no explanation.\n\nSummary: {summary}",
    max_tokens=100
)
search_keywords_clean = keywords.replace("\n", "").strip()
print(f"\n✅ Keywords: {search_keywords_clean}")

with open("outputs/stage5_keywords.txt", "w") as f:
    f.write(search_keywords_clean)
print("💾 Saved → outputs/stage5_keywords.txt")

In [ ]:
# Stage 7f: Fetch LinkedIn jobs
import json

print("💼 Fetching LinkedIn jobs...")
linkedin_jobs = fetch_linkedin_jobs(search_keywords_clean, rows=60)
print(f"✅ Found {len(linkedin_jobs)} LinkedIn jobs")

if linkedin_jobs:
    print("\n--- Top 5 LinkedIn Jobs ---")
    for job in linkedin_jobs[:5]:
        print(f"  • {job.get('title')} at {job.get('companyName')} | {job.get('location')}")

with open("outputs/stage6_linkedin_jobs.json", "w") as f:
    json.dump(linkedin_jobs, f, indent=2)
print("\n💾 Saved → outputs/stage6_linkedin_jobs.json")

In [ ]:
# Stage 7g: Fetch Naukri jobs
print("💼 Fetching Naukri jobs...")
naukri_jobs = fetch_naukri_jobs(search_keywords_clean, rows=60)
print(f"✅ Found {len(naukri_jobs)} Naukri jobs")

if naukri_jobs:
    print("\n--- Top 5 Naukri Jobs ---")
    for job in naukri_jobs[:5]:
        print(f"  • {job.get('title')} at {job.get('companyName')} | {job.get('location')}")

with open("outputs/stage7_naukri_jobs.json", "w") as f:
    json.dump(naukri_jobs, f, indent=2)
print("\n💾 Saved → outputs/stage7_naukri_jobs.json")

## 📋 Stage 8 — Write `requirements.txt` & `README.md`

In [ ]:
# Stage 8a: Write requirements.txt
req = """streamlit
openai
pymupdf
python-dotenv
apify-client
mcp[cli]
"""
with open("requirements.txt", "w") as f:
    f.write(req)
print("✅ requirements.txt written")

In [ ]:
# Stage 8b: Write README.md
readme = """# 📄 AI Job Recommender

An AI-powered resume analyzer and job recommender using OpenAI GPT-4o, Apify scrapers for LinkedIn & Naukri, and Streamlit.

## Features
- Resume text extraction (PDF)
- AI-powered resume summary, skill gap analysis, and career roadmap
- Live job recommendations from LinkedIn and Naukri (India)
- MCP tool server for agentic workflows

## Setup
1. Clone the repo
2. Create a `.env` file with:
   ```
   OPENAI_API_KEY=your-key
   APIFY_API_TOKEN=your-token
   ```
3. Install dependencies: `pip install -r requirements.txt`
4. Run: `streamlit run app.py`

## Project Structure
```
├── app.py                  # Streamlit frontend
├── mcp_server.py           # MCP tool server
├── requirements.txt
├── src/
│   ├── helper.py           # PDF extraction + OpenAI calls
│   └── job_api.py          # Apify LinkedIn & Naukri scrapers
└── outputs/                # Pipeline outputs (Colab)
```
"""
with open("README.md", "w") as f:
    f.write(readme)
print("✅ README.md written")

## 📦 Stage 9 — Zip & Download All Outputs

In [ ]:
# Stage 9: Zip everything and download
import zipfile, os
from google.colab import files

zip_path = "AI_Job_Recommender_Outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    # Project source files
    for fname in ["app.py", "mcp_server.py", "requirements.txt", "README.md"]:
        if os.path.exists(fname):
            zf.write(fname)
    # src/ folder
    for root, dirs, fnames in os.walk("src"):
        for fn in fnames:
            fpath = os.path.join(root, fn)
            zf.write(fpath)
    # outputs/ folder
    for root, dirs, fnames in os.walk("outputs"):
        for fn in fnames:
            fpath = os.path.join(root, fn)
            zf.write(fpath)

print(f"✅ Zipped: {zip_path}")
print("\n📂 Contents:")
with zipfile.ZipFile(zip_path, "r") as zf:
    for name in zf.namelist():
        print(f"  {name}")

files.download(zip_path)
print("\n⬇️ Download started!")